# Data Manipulation
- Menggunakan  type data dengan benar menggunakan type conversion
- Melakukan filtering dan sorting data berdasarkan kondisi tertentu.
- Menggunakan groupby() untuk melakukan agregasi (misal rata-rata penjualan per kategori).
- Membuat kolom baru melalui feature engineering 
- Menormalisasikan data yang mengandung outlier menggunakan Scaling

In [35]:
# Membuat salinan data untuk proses analisis
df_analysis = df.copy()

# Memastikan tipe data sudah sesuai
df_analysis['Quantity'] = df_analysis['Quantity'].astype(int)
df_analysis['Price Per Unit'] = df_analysis['Price Per Unit'].astype(float)
df_analysis['Total Spent'] = df_analysis['Total Spent'].astype(float)
df_analysis['Transaction Date'] = pd.to_datetime(df_analysis['Transaction Date'])

# Menampilkan tipe data akhir
df_analysis.dtypes

Transaction ID              object
Item                        object
Quantity                     int64
Price Per Unit             float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
dtype: object

Feature engineering dilakukan dengan membuat kolom baru dari Transaction Date, yaitu Year, Month, Month Name, dan Day Name. 
Selain itu, dibuat kolom Spending Category untuk mengelompokkan transaksi berdasarkan nilai Total Spent.

In [36]:
# Membuat kolom baru berdasarkan tanggal transaksi
df_analysis['Year'] = df_analysis['Transaction Date'].dt.year
df_analysis['Month'] = df_analysis['Transaction Date'].dt.month
df_analysis['Month Name'] = df_analysis['Transaction Date'].dt.month_name()
df_analysis['Day Name'] = df_analysis['Transaction Date'].dt.day_name()

# Membuat kategori transaksi berdasarkan Total Spent
df_analysis['Spending Category'] = pd.cut(
    df_analysis['Total Spent'],
    bins=[0, 5, 10, 20, df_analysis['Total Spent'].max()],
    labels=['Low', 'Medium', 'High', 'Very High'],
    include_lowest=True
)

df_analysis.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Year,Month,Month Name,Day Name,Spending Category
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,2023,9,September,Friday,Low
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,2023,5,May,Tuesday,High
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,2023,7,July,Wednesday,Low
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,In-store,2023-04-27,2023,4,April,Thursday,Medium
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,2023,6,June,Sunday,Low


#### Scaling data numerik

Scaling dilakukan menggunakan MinMaxScaler untuk mengubah nilai numerik ke dalam rentang 0 sampai 1. 

Kolom yang dinormalisasi adalah Quantity, Price Per Unit, dan Total Spent. Scaling dilakukan agar perbedaan skala antar kolom numerik tidak terlalu besar, terutama setelah ditemukan outlier pada Total Spent.

In [45]:
from sklearn.preprocessing import MinMaxScaler

# Membuat salinan dari df_analysis agar kolom hasil feature engineering tetap terbawa
df_scaled = df_analysis.copy()

# Kolom numerik yang akan dinormalisasi
columns_to_scale = ['Quantity', 'Price Per Unit', 'Total Spent']

# Membuat data numerik yang sudah dicapping
df_numeric_capped = df_analysis[columns_to_scale].copy()

for col in columns_to_scale:
    Q1 = df_numeric_capped[col].quantile(0.25)
    Q3 = df_numeric_capped[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_numeric_capped[col] = df_numeric_capped[col].clip(lower_bound, upper_bound)

# Scaling
scaler = MinMaxScaler()

df_scaled[['Quantity Scaled', 'Price Per Unit Scaled', 'Total Spent Scaled']] = scaler.fit_transform(
    df_numeric_capped[columns_to_scale]
)

df_scaled.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Year,Month,Month Name,Day Name,Spending Category,Quantity Scaled,Price Per Unit Scaled,Total Spent Scaled
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,2023,9,September,Friday,Low,0.25,0.25,0.130435
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,2023,5,May,Tuesday,High,0.75,0.50,0.478261
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,2023,7,July,Wednesday,Low,0.75,0.00,0.130435
3,TXN_7034554,Salad,2,5.0,10.0,Digital Wallet,In-store,2023-04-27,2023,4,April,Thursday,Medium,0.25,1.00,0.391304
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,2023,6,June,Sunday,Low,0.25,0.25,0.130435


Sebelum masuk ke tahap categorical data encoding, dilakukan pengecekan ulang terhadap dataset hasil data manipulation. Dataset sudah tidak memiliki missing value, tipe data sudah sesuai, dan beberapa kolom baru hasil feature engineering telah dibuat. 

Selanjutnya, atribut kategorikal akan diubah menjadi bentuk numerik agar dapat digunakan untuk analisis lanjutan.

In [46]:
# Mengecek kolom kategorikal
categorical_columns = df_scaled.select_dtypes(include=['object', 'category']).columns

categorical_columns

Index(['Transaction ID', 'Item', 'Payment Method', 'Location', 'Month Name',
       'Day Name', 'Spending Category'],
      dtype='object')

In [47]:
# Menghapus Transaction ID dari kolom yang akan di-encoding
categorical_columns = categorical_columns.drop('Transaction ID')

categorical_columns

Index(['Item', 'Payment Method', 'Location', 'Month Name', 'Day Name',
       'Spending Category'],
      dtype='object')

#### Membuat Ordinal encoding untuk Spending Category

In [48]:
# Membuat salinan data untuk proses encoding
df_encoding = df_scaled.copy()

# Mapping ordinal untuk Spending Category
spending_mapping = {
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Very High': 4
}

# Membuat kolom baru hasil ordinal encoding
df_encoding['Spending Category Encoded'] = df_encoding['Spending Category'].map(spending_mapping)

df_encoding[['Spending Category', 'Spending Category Encoded']].head()

,Spending Category,Spending Category Encoded
0,Low,1
1,High,3
2,Low,1
3,Medium,2
4,Low,1


#### One-hot Encoding

Encoding ini akan digunakan untuk kategori lainnya

In [49]:
# Menentukan kolom yang akan dilakukan one-hot encoding
columns_to_encode = [
    'Item',
    'Payment Method',
    'Location',
    'Month Name',
    'Day Name'
]

# Melakukan one-hot encoding
df_encoded = pd.get_dummies(
    df_encoding,
    columns=columns_to_encode,
    drop_first=False
)

df_encoded.head()

,Transaction ID,Quantity,Price Per Unit,Total Spent,Transaction Date,Year,Month,Spending Category,Quantity Scaled,Price Per Unit Scaled,...,Month Name_November,Month Name_October,Month Name_September,Day Name_Friday,Day Name_Monday,Day Name_Saturday,Day Name_Sunday,Day Name_Thursday,Day Name_Tuesday,Day Name_Wednesday
0,TXN_1961373,2,2.0,4.0,2023-09-08,2023,9,Low,0.25,0.25,...,False,False,True,True,False,False,False,False,False,False
1,TXN_4977031,4,3.0,12.0,2023-05-16,2023,5,High,0.75,0.50,...,False,False,False,False,False,False,False,False,True,False
2,TXN_4271903,4,1.0,4.0,2023-07-19,2023,7,Low,0.75,0.00,...,False,False,False,False,False,False,False,False,False,True
3,TXN_7034554,2,5.0,10.0,2023-04-27,2023,4,Medium,0.25,1.00,...,False,False,False,False,False,False,False,True,False,False
4,TXN_3160411,2,2.0,4.0,2023-06-11,2023,6,Low,0.25,0.25,...,False,False,False,False,False,False,True,False,False,False


Mengubah hasil encoding menjadi 0/1 dari True/false

In [50]:
# Mengubah kolom boolean hasil encoding menjadi integer 0 dan 1

bool_columns = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_columns] = df_encoded[bool_columns].astype(int)

df_encoded.head()

,Transaction ID,Quantity,Price Per Unit,Total Spent,Transaction Date,Year,Month,Spending Category,Quantity Scaled,Price Per Unit Scaled,...,Month Name_November,Month Name_October,Month Name_September,Day Name_Friday,Day Name_Monday,Day Name_Saturday,Day Name_Sunday,Day Name_Thursday,Day Name_Tuesday,Day Name_Wednesday
0,TXN_1961373,2,2.0,4.0,2023-09-08,2023,9,Low,0.25,0.25,...,0,0,1,1,0,0,0,0,0,0
1,TXN_4977031,4,3.0,12.0,2023-05-16,2023,5,High,0.75,0.50,...,0,0,0,0,0,0,0,0,1,0
2,TXN_4271903,4,1.0,4.0,2023-07-19,2023,7,Low,0.75,0.00,...,0,0,0,0,0,0,0,0,0,1
3,TXN_7034554,2,5.0,10.0,2023-04-27,2023,4,Medium,0.25,1.00,...,0,0,0,0,0,0,0,1,0,0
4,TXN_3160411,2,2.0,4.0,2023-06-11,2023,6,Low,0.25,0.25,...,0,0,0,0,0,0,1,0,0,0


In [51]:
# Mengecek hasil encoding

print("Jumlah kolom sebelum encoding:", df_scaled.shape[1])
print("Jumlah kolom setelah encoding:", df_encoded.shape[1])

print("\nKolom hasil encoding:")
print(df_encoded.columns)

Jumlah kolom sebelum encoding: 16
Jumlah kolom setelah encoding: 44

Kolom hasil encoding:
Index(['Transaction ID', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Transaction Date', 'Year', 'Month', 'Spending Category',
       'Quantity Scaled', 'Price Per Unit Scaled', 'Total Spent Scaled',
       'Spending Category Encoded', 'Item_Cake', 'Item_Coffee', 'Item_Cookie',
       'Item_Juice', 'Item_Salad', 'Item_Sandwich', 'Item_Smoothie',
       'Item_Tea', 'Payment Method_Cash', 'Payment Method_Credit Card',
       'Payment Method_Digital Wallet', 'Location_In-store',
       'Location_Takeaway', 'Month Name_April', 'Month Name_August',
       'Month Name_December', 'Month Name_February', 'Month Name_January',
       'Month Name_July', 'Month Name_June', 'Month Name_March',
       'Month Name_May', 'Month Name_November', 'Month Name_October',
       'Month Name_September', 'Day Name_Friday', 'Day Name_Monday',
       'Day Name_Saturday', 'Day Name_Sunday', 'Day Name_Thursday',
    

In [52]:
# Pengecekan missing value setelah encoding

df_encoded.isnull().sum()

Transaction ID                   0
Quantity                         0
Price Per Unit                   0
Total Spent                      0
Transaction Date                 0
Year                             0
Month                            0
Spending Category                0
Quantity Scaled                  0
Price Per Unit Scaled            0
Total Spent Scaled               0
Spending Category Encoded        0
Item_Cake                        0
Item_Coffee                      0
Item_Cookie                      0
Item_Juice                       0
Item_Salad                       0
Item_Sandwich                    0
Item_Smoothie                    0
Item_Tea                         0
Payment Method_Cash              0
Payment Method_Credit Card       0
Payment Method_Digital Wallet    0
Location_In-store                0
Location_Takeaway                0
Month Name_April                 0
Month Name_August                0
Month Name_December              0
Month Name_February 

In [53]:
# Tampilan data awal hasil encoding

df_encoded.head()

,Transaction ID,Quantity,Price Per Unit,Total Spent,Transaction Date,Year,Month,Spending Category,Quantity Scaled,Price Per Unit Scaled,...,Month Name_November,Month Name_October,Month Name_September,Day Name_Friday,Day Name_Monday,Day Name_Saturday,Day Name_Sunday,Day Name_Thursday,Day Name_Tuesday,Day Name_Wednesday
0,TXN_1961373,2,2.0,4.0,2023-09-08,2023,9,Low,0.25,0.25,...,0,0,1,1,0,0,0,0,0,0
1,TXN_4977031,4,3.0,12.0,2023-05-16,2023,5,High,0.75,0.50,...,0,0,0,0,0,0,0,0,1,0
2,TXN_4271903,4,1.0,4.0,2023-07-19,2023,7,Low,0.75,0.00,...,0,0,0,0,0,0,0,0,0,1
3,TXN_7034554,2,5.0,10.0,2023-04-27,2023,4,Medium,0.25,1.00,...,0,0,0,0,0,0,0,1,0,0
4,TXN_3160411,2,2.0,4.0,2023-06-11,2023,6,Low,0.25,0.25,...,0,0,0,0,0,0,1,0,0,0


In [54]:
# Membuat dataset final numerik setelah encoding
df_model = df_encoded.drop(
    columns=['Transaction ID', 'Transaction Date', 'Spending Category']
)

df_model.head()

,Quantity,Price Per Unit,Total Spent,Year,Month,Quantity Scaled,Price Per Unit Scaled,Total Spent Scaled,Spending Category Encoded,Item_Cake,...,Month Name_November,Month Name_October,Month Name_September,Day Name_Friday,Day Name_Monday,Day Name_Saturday,Day Name_Sunday,Day Name_Thursday,Day Name_Tuesday,Day Name_Wednesday
0,2,2.0,4.0,2023,9,0.25,0.25,0.130435,1,0,...,0,0,1,1,0,0,0,0,0,0
1,4,3.0,12.0,2023,5,0.75,0.50,0.478261,3,1,...,0,0,0,0,0,0,0,0,1,0
2,4,1.0,4.0,2023,7,0.75,0.00,0.130435,1,0,...,0,0,0,0,0,0,0,0,0,1
3,2,5.0,10.0,2023,4,0.25,1.00,0.391304,2,0,...,0,0,0,0,0,0,0,1,0,0
4,2,2.0,4.0,2023,6,0.25,0.25,0.130435,1,0,...,0,0,0,0,0,0,1,0,0,0


In [55]:
# Mengecek tipe data pada dataset final
df_model.dtypes

Quantity                            int64
Price Per Unit                    float64
Total Spent                       float64
Year                                int32
Month                               int32
Quantity Scaled                   float64
Price Per Unit Scaled             float64
Total Spent Scaled                float64
Spending Category Encoded        category
Item_Cake                           int64
Item_Coffee                         int64
Item_Cookie                         int64
Item_Juice                          int64
Item_Salad                          int64
Item_Sandwich                       int64
Item_Smoothie                       int64
Item_Tea                            int64
Payment Method_Cash                 int64
Payment Method_Credit Card          int64
Payment Method_Digital Wallet       int64
Location_In-store                   int64
Location_Takeaway                   int64
Month Name_April                    int64
Month Name_August                 

In [56]:
# Mengecek apakah masih ada kolom object, category, atau datetime
df_model.select_dtypes(include=['object', 'category', 'datetime']).columns

Index(['Spending Category Encoded'], dtype='object')

Setelah proses categorical data encoding, beberapa kolom kategorikal telah diubah menjadi bentuk numerik. Kolom Spending Category diubah menggunakan ordinal encoding karena memiliki tingkatan dari Low sampai Very High. Sementara itu, kolom Item, Payment Method, Location, Month Name, dan Day Name diubah menggunakan one-hot encoding karena tidak memiliki urutan tertentu.

Hasil encoding membuat dataset lebih siap digunakan untuk analisis lanjutan karena seluruh atribut kategorikal sudah direpresentasikan dalam bentuk numerik.